In [ ]:
!pip install librosa timm -q

import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings("ignore")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
DATA_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
SR = 32000
DURATION = 6
SAMPLES = SR * DURATION
BATCH_SIZE = 32
EPOCHS = 12
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GENRES = ["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]
label2idx = {g:i for i,g in enumerate(GENRES)}
idx2label = {i:g for g,i in label2idx.items()}

In [ ]:
data = []

for genre in GENRES:
    genre_path = os.path.join(DATA_PATH, "genres_stems", genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        stems = {
            "drums": os.path.join(song_path, "drums.wav"),
            "vocals": os.path.join(song_path, "vocals.wav"),
            "bass": os.path.join(song_path, "bass.wav"),
            "other": os.path.join(song_path, "other.wav"),
        }
        data.append((genre, stems))

df = pd.DataFrame(data, columns=["genre", "stems"])
df["label"] = df["genre"].map(label2idx)

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

In [ ]:
import soundfile as sf
import librosa
import numpy as np

def load_audio(path):
    y, sr = sf.read(path)

    if len(y.shape) > 1:
        y = y.mean(axis=1)

    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)

    if len(y) > SAMPLES:
        y = y[:SAMPLES]
    else:
        y = np.pad(y, (0, SAMPLES - len(y)))

    y = y / (np.max(np.abs(y)) + 1e-6)

    return y


def mel_spec(y):
    spec = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_mels=128,
        n_fft=2048,
        hop_length=512
    )

    spec = librosa.power_to_db(spec)

    delta = librosa.feature.delta(spec)
    delta2 = librosa.feature.delta(spec, order=2)

    spec = np.stack([spec, delta, delta2], axis=0).astype(np.float32)

    return spec

In [ ]:
class MashupDataset(Dataset):
    def __init__(self, df, noise_files):
        self.df = df.reset_index(drop=True)
        self.noise_files = noise_files

        self.genre_groups = {
            g: df[df["genre"] == g].reset_index(drop=True)
            for g in df["genre"].unique()
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        genre = self.df.iloc[idx]["genre"]
        subset = self.genre_groups[genre]

        stems = {}
        for stem in ["drums","vocals","bass","other"]:
            sample = subset.sample(1).iloc[0]["stems"][stem]
            stems[stem] = load_audio(sample)

        mix = (
            random.uniform(0.2,1.8) * stems["drums"] +
            random.uniform(0.2,1.8) * stems["vocals"] +
            random.uniform(0.1,1.5) * stems["bass"] +
            random.uniform(0.1,1.5) * stems["other"]
        )

        if random.random() < 0.6:
            mix *= random.uniform(0.4, 2.0)

        # 🔥 slightly reduced noise probability (more stable)
        if random.random() < 0.8:
            noise = load_audio(random.choice(self.noise_files))
            mix += random.uniform(1.0, 2.5) * noise

        shift = random.randint(0, int(0.5 * len(mix)))
        mix = np.roll(mix, shift)

        if random.random() < 0.5:
            start = random.randint(0, len(mix)//2)
            length = random.randint(1000, 4000)
            mix[start:start+length] = 0

        if len(mix) > SAMPLES:
            start = random.randint(0, len(mix) - SAMPLES)
            mix = mix[start:start+SAMPLES]
        else:
            mix = np.pad(mix, (0, SAMPLES - len(mix)))

        mix = mix / (np.max(np.abs(mix)) + 1e-6)

        spec = mel_spec(mix)

        label = label2idx[genre]

        return torch.tensor(spec).float(), torch.tensor(label, dtype=torch.long)

In [ ]:
noise_dir = os.path.join(DATA_PATH, "ESC-50-master/audio")
noise_files = [os.path.join(noise_dir, f) for f in os.listdir(noise_dir)]

In [ ]:
train_dataset = MashupDataset(train_df, noise_files)
val_dataset = MashupDataset(val_df, noise_files)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,   
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
class ModelA(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model("efficientnet_b0", pretrained=True, in_chans=3, num_classes=10)
    def forward(self, x):
        return self.model(x)

# class ModelB(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.model = timm.create_model("resnet18", pretrained=True, in_chans=3, num_classes=10)
#     def forward(self, x):
#         return self.model(x)

class ModelC(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model("mobilenetv3_small_100", pretrained=True, in_chans=3, num_classes=10)

    def forward(self, x):
         return self.model(x)

class ModelD(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model("efficientnet_b2", pretrained=True, in_chans=3, num_classes=10)
    def forward(self, x):
        return self.model(x)

In [ ]:
import torch
from sklearn.metrics import f1_score

torch.backends.cudnn.benchmark = True 

modelA = ModelA().to(DEVICE)   # EfficientNet
modelC = ModelC().to(DEVICE)   # MobileNet
modelD = ModelD().to(DEVICE)   # DenseNet

optimizerA = torch.optim.AdamW(modelA.parameters(), lr=LR)
optimizerC = torch.optim.AdamW(modelC.parameters(), lr=LR)
optimizerD = torch.optim.AdamW(modelD.parameters(), lr=LR)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

schedulerA = torch.optim.lr_scheduler.CosineAnnealingLR(optimizerA, T_max=EPOCHS)
schedulerC = torch.optim.lr_scheduler.CosineAnnealingLR(optimizerC, T_max=EPOCHS)
schedulerD = torch.optim.lr_scheduler.CosineAnnealingLR(optimizerD, T_max=EPOCHS)

scalerA = torch.cuda.amp.GradScaler()
scalerC = torch.cuda.amp.GradScaler()
scalerD = torch.cuda.amp.GradScaler()

best_A, best_C, best_D = 0, 0, 0

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    modelA.train()
    modelC.train()
    modelD.train()

    for x, y in train_loader:
        x = x.to(DEVICE).float()   # 🔥 fix
        y = y.to(DEVICE).long()

        # ===== EfficientNet =====
        optimizerA.zero_grad()
        with torch.cuda.amp.autocast():
            outA = modelA(x)
            lossA = criterion(outA, y)
        scalerA.scale(lossA).backward()
        torch.nn.utils.clip_grad_norm_(modelA.parameters(), 1.0)
        scalerA.step(optimizerA)
        scalerA.update()

        # ===== MobileNet =====
        optimizerC.zero_grad()
        with torch.cuda.amp.autocast():
            outC = modelC(x)
            lossC = criterion(outC, y)
        scalerC.scale(lossC).backward()
        torch.nn.utils.clip_grad_norm_(modelC.parameters(), 1.0)
        scalerC.step(optimizerC)
        scalerC.update()

        # ===== DenseNet =====
        optimizerD.zero_grad()
        with torch.cuda.amp.autocast():
            outD = modelD(x)
            lossD = criterion(outD, y)
        scalerD.scale(lossD).backward()
        torch.nn.utils.clip_grad_norm_(modelD.parameters(), 1.0)
        scalerD.step(optimizerD)
        scalerD.update()

    schedulerA.step()
    schedulerC.step()
    schedulerD.step()

    modelA.eval()
    modelC.eval()
    modelD.eval()

    predsA, predsC, predsD, targets = [], [], [], []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE).float()  

            outA = modelA(x)
            outC = modelC(x)
            outD = modelD(x)

            predsA.extend(outA.argmax(1).cpu().numpy())
            predsC.extend(outC.argmax(1).cpu().numpy())
            predsD.extend(outD.argmax(1).cpu().numpy())

            targets.extend(y.cpu().numpy())   

    f1A = f1_score(targets, predsA, average="macro")
    f1C = f1_score(targets, predsC, average="macro")
    f1D = f1_score(targets, predsD, average="macro")

    if f1A > best_A:
        best_A = f1A
        torch.save(modelA.state_dict(), "best_A.pth")

    if f1C > best_C:
        best_C = f1C
        torch.save(modelC.state_dict(), "best_C.pth")

    if f1D > best_D:
        best_D = f1D
        torch.save(modelD.state_dict(), "best_D.pth")

    print(epoch, round(f1A,4), round(f1C,4), round(f1D,4))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score

class CNN_LSTM_Scratch(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.fc1 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        B, C, H, W = x.shape
        x = x.mean(dim=2)
        x = x.permute(0, 2, 1)

        x, _ = self.lstm(x)
        x = x.mean(dim=1)

        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)

        return x


# ===== INIT =====
modelE = CNN_LSTM_Scratch().to(DEVICE)
optimizerE = torch.optim.AdamW(modelE.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

best_E = 0

# ===== TRAIN LOOP =====
for epoch in range(EPOCHS):

    modelE.train()

    for x, y in train_loader:
        x = x.to(DEVICE).float()     
        y = y.to(DEVICE).long()

        optimizerE.zero_grad()
        out = modelE(x)
        loss = criterion(out, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(modelE.parameters(), 1.0) 

        optimizerE.step()

    # ===== VALIDATION =====
    modelE.eval()
    preds, targets = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE).float()   

            out = modelE(x)
            preds.extend(out.argmax(1).cpu().numpy())
            targets.extend(y.cpu().numpy())  

    f1 = f1_score(targets, preds, average="macro")

    if f1 > best_E:
        best_E = f1
        torch.save(modelE.state_dict(), "best_E.pth")

    print(epoch, round(f1, 4))

In [ ]:
import random
import numpy as np
import torch
import pandas as pd
import os

modelA = ModelA().to(DEVICE)
modelC = ModelC().to(DEVICE)
modelD = ModelD().to(DEVICE)

modelA.load_state_dict(torch.load("best_A.pth", map_location=DEVICE))
modelC.load_state_dict(torch.load("best_C.pth", map_location=DEVICE))
modelD.load_state_dict(torch.load("best_D.pth", map_location=DEVICE))

modelA.eval()
modelC.eval()
modelD.eval()

test_df = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

predictions = []

with torch.no_grad():

    for _, row in test_df.iterrows():
        filename = "song" + str(row["id"]).zfill(4) + ".wav"
        path = os.path.join(DATA_PATH, "mashups", filename)

        y = load_audio(path)

        outputs = []

        for _ in range(12):

            if len(y) > SAMPLES:
                start = random.randint(0, len(y) - SAMPLES)
                crop = y[start:start+SAMPLES]
            else:
                crop = np.pad(y, (0, SAMPLES - len(y)))

            if random.random() < 0.5:
                crop = crop[::-1]

            shift = random.randint(0, int(0.25 * len(crop)))
            crop = np.roll(crop, shift)

            s = mel_spec(crop)
            s = torch.tensor(s).float().unsqueeze(0).to(DEVICE)

            outA = torch.softmax(modelA(s), dim=1)
            outC = torch.softmax(modelC(s), dim=1)
            outD = torch.softmax(modelD(s), dim=1)

            out = ( 0.4 * outD + 0.35 * outC + 0.25 * outA )

            outputs.append(out)

        out = torch.mean(torch.stack(outputs), dim=0)
        pred = out.argmax(1).item()

        predictions.append(pred)

predictions = [idx2label[p] for p in predictions]

sub = pd.DataFrame({
    "id": test_df["id"].apply(lambda x: str(x).zfill(4)),
    "genre": predictions
})

sub.to_csv("submission.csv", index=False)